Install dependencies and imports

In [ ]:
!pip install -q transformers rdkit

import os, time
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from scipy.stats import pearsonr, spearmanr

torch.manual_seed(0); np.random.seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| GPU:', torch.cuda.get_device_name(0) if device=='cuda' else 'none')

In [ ]:
!pip show transformers

The data TSV (?) saves to Google Drive (for example `MyDrive/dti/data.tsv`). 

# TODO OH: what disk do you store databases on? 

# TODO OH: what type of data files have you explored and do you like? (look at chatted chart in notes)

The embeddings and checkpoints should be saved under `CACHE`. Ensures that they're computed once and then load back almost instantly the next time the notebook gets run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

DATA_TSV  = '/content/drive/MyDrive/dti/pbcnet2_bindingdb_combined_uniprot_with_splits.tsv'  # edit if needed
CACHE = '/content/drive/MyDrive/dti/cache'
os.makedirs(CACHE, exist_ok=True)

assert os.path.exists(DATA_TSV), f'TSV not found at {DATA_TSV} — upload it to Drive or fix the path.'
print('ok')

# load data from drive (? Q above)

In [ ]:
### load data and visually inspect things went smoothly :)  
df = pd.read_csv(DATA_TSV, sep='\t')
df.head()

# Set data up and run cleaning assurances

In [ ]:
df.columns = ['binder smiles', 'recepetor seq', 'aff', 'receptor uniprot','receptor family','binder type', 'data source', 'split'] # rename column names for simplicity

df['seq'] = df['seq'].astype(str).str.upper().str.strip()      # ESM-2 needs uppercase
df['aff'] = pd.to_numeric(df['aff'], errors='coerce')          # make sure all the aff labels are numerical values, coerce reverts to NaN
df = df.dropna(subset=['smiles', 'seq', 'aff'])                # cleaning, drop nan in smiles, seq, aff columns (insufficient training info), shouldn't happen in our case (hand curated data set)
df = df[df['aff'] > 0]                                         # cleaning, only positive affinity value makes sense
df['y'] = 9.0 - np.log10(df['aff'].values)                     # nM -> p-scale

SUBSAMPLE_ROWS = 2000
if SUBSAMPLE_ROWS:                                             # draw 80/10/10 from the existing splits
    target = {'train': 0.8, 'valid': 0.1, 'test': 0.1}
    parts = []
    for split_group, portion in target.items():
        g = df[df.split == split_group]
        n = min(len(g), int(round(SUBSAMPLE_ROWS * portion)))   # catches if there are fewer than subsample rows*target portion
        parts.append(g.sample(n, random_state=0))               
    df = pd.concat(parts)

train_df = df[df.split == 'train']
valid_df = df[df.split == 'valid']
test_df  = df[df.split == 'test']

#report data metrics
print({k: len(v) for k, v in [('train', train_df), ('valid', valid_df), ('test', test_df)]})
print('unique drugs:', df.smiles.nunique(), '| unique proteins:', df.seq.nunique())
print('pAffinity: min %.2f  median %.2f  max %.2f' % (df.y.min(), df.y.median(), df.y.max()))

# TODO OH: labeling noise ? 

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(2, 2, figsize=(12, 8))

# 1) overall pAffinity
ax[0,0].hist(df.y, bins=60, color='#4C72B0', edgecolor='white')
ax[0,0].axvline(df.y.median(), color='crimson', ls='--', label=f'median {df.y.median():.2f}')
ax[0,0].set(title='pAffinity distribution', xlabel='pAffinity = 9 - log10(nM)', ylabel='count'); ax[0,0].legend()

# 2) pAffinity density by split
for sp, c in [('train','#4C72B0'), ('valid','#DD8452'), ('test','#55A868')]:
    d = df[df.split == sp].y
    if len(d):
        ax[0,1].hist(d, bins=50, density=True, histtype='step', lw=2, color=c, label=f'{sp} (n={len(d)})')
ax[0,1].set(title='pAffinity density by split', xlabel='pAffinity'); ax[0,1].legend()

# 3) pollutant type mix
pt = df.pollutant_type.value_counts()
ax[1,0].bar(pt.index, pt.values, color='#8172B3')
ax[1,0].set(title = 'binder type', ylabel='count')

## TODO : add rest of relevant visualization 

# 4) protein length (unique proteins), clipped for readability
plens = df.drop_duplicates('seq').seq.str.len()
ax[1,1].hist(plens.clip(upper=2000), bins=60, color='#937860', edgecolor='white')
ax[1,1].axvline(1024, color='crimson', ls='--', label='ESM-2 truncation (1024)')
ax[1,1].set(title='protein length (unique, clipped at 2000)', xlabel='residues', ylabel='count'); ax[1,1].legend()

plt.tight_layout(); plt.show()

frac_trunc = (plens > 1024).mean()
print(f'{frac_trunc:.1%} of unique proteins exceed 1024 residues and get truncated')

# embed

In [ ]:
from transformers import AutoTokenizer, AutoModel

def load_frozen(name):
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModel.from_pretrained(name).to(device).eval()
    for p in model.parameters():
        p.requires_grad = False
    return tok, model

esm_tok,  esm_model  = load_frozen('facebook/esm2_t6_8M_UR50D')      # protein
chem_tok, chem_model = load_frozen('DeepChem/ChemBERTa-77M-MLM')     # drug

PROT_DIM = esm_model.config.hidden_size
DRUG_DIM = chem_model.config.hidden_size
print(f'protein dim = {PROT_DIM}, drug dim = {DRUG_DIM}')

enhance binding data (endonet, add known pollutants as well) 

This is the primary place to enrich information as possible. Need to cover ALL forms of estrogens/androgens, not just simple ones... see if there's ptm data even... ptms will be more significant on smaller structures like hormones. call with Fiona and Sophie @ PBG to consult on data curation 



# build MLP 

# train

# test

need a test set of validated endo disrupting 

curate applicaitons set 

SYNTHETIC FRAGRENCES, unregistered cosmetic products, continue researching, known inhibitors (spironolactine), endongenous hormones (all forms) 